In [4]:
import sys
import torch

sys.path.insert(0, ".")
from DialogueGraph import (
    _SelfAttentivePooling,
    _UtteranceFeatureExtractor,
    _DialogueGCNLayer,
    _DialogueGCN,
    _ContextAttentionPooling,
    DialogueGraph,
)

from transformers import (
    BertModel,
    BertTokenizerFast,
    WavLMModel,
    Wav2Vec2Processor,
)

SAMPLE_RATE = 16_000
B, T = 2, 4
D_FEAT = 64
D_OUT  = 256
N_CTX  = T - 1

bert_name   = "bert-base-uncased"
wavlm_name  = "microsoft/wavlm-base"

real_bert      = BertModel.from_pretrained(bert_name)
real_tokenizer = BertTokenizerFast.from_pretrained(bert_name)
real_wavlm     = WavLMModel.from_pretrained(wavlm_name)
real_processor = Wav2Vec2Processor.from_pretrained(wavlm_name)

real_bert.eval()
real_wavlm.eval()

print("Encoder models loaded.")


c:\Users\jackm\miniconda3\envs\capstone-eda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\jackm\miniconda3\envs\capstone-eda\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jackm\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to ac

KeyboardInterrupt: 

In [2]:
pool = _SelfAttentivePooling(dim=D_FEAT)

# basic shape: (B, T, d) -> (B, d)
x    = torch.randn(B, T, D_FEAT)
out  = pool(x)
assert out.shape == (B, D_FEAT), f"expected ({B}, {D_FEAT}), got {out.shape}"

# with mask: same output shape, fully-masked rows should be zero
mask = torch.ones(B, T)
mask[0, :] = 0          # batch item 0 is fully masked
out_masked  = pool(x, mask)
assert out_masked.shape == (B, D_FEAT)
assert torch.allclose(out_masked[0], torch.zeros(D_FEAT)), "fully-masked row should be zero"
assert not torch.allclose(out_masked[1], torch.zeros(D_FEAT)), "unmasked row should be non-zero"

print("PASS _SelfAttentivePooling")


PASS _SelfAttentivePooling


In [ ]:
import sys
import torch

sys.path.insert(0, ".")
from DialogueGraph import (
    _SelfAttentivePooling,
    _UtteranceFeatureExtractor,
    _DialogueGCNLayer,
    _DialogueGCN,
    _ContextAttentionPooling,
    DialogueGraph,
)

from transformers import (
    BertModel,
    BertTokenizerFast,
    WavLMModel,
    Wav2Vec2FeatureExtractor,
)


SAMPLE_RATE = 16_000
B, T = 2, 4
D_FEAT = 64
D_OUT  = 256
N_CTX  = T - 1

bert_name   = "bert-base-uncased"
wavlm_name  = "microsoft/wavlm-base"

real_bert      = BertModel.from_pretrained(bert_name)
real_tokenizer = BertTokenizerFast.from_pretrained(bert_name)
real_wavlm     = WavLMModel.from_pretrained(wavlm_name)
real_processor = Wav2Vec2FeatureExtractor.from_pretrained(wavlm_name)


real_bert.eval()
real_wavlm.eval()

print("Encoder models loaded.")


In [ ]:
# 2-speaker conversation: A B A B (4 context turns)
spk  = [["A", "B", "A", "B"], ["A", "A", "B", "B"]]
N    = 4
adj  = _DialogueGCNLayer.build_adjacency(spk, N, device=torch.device("cpu"))

assert adj.shape == (2, _DialogueGCNLayer.NUM_RELATIONS, N, N), \
    f"expected (2, 5, {N}, {N}), got {adj.shape}"

# self-loops must be on for every node
self_rel = _DialogueGCNLayer._SELF
assert adj[0, self_rel].diag().all(), "self-loops missing in batch 0"
assert adj[1, self_rel].diag().all(), "self-loops missing in batch 1"

# off-diagonal entries must not appear in the SELF relation
off_diag_mask = ~torch.eye(N, dtype=torch.bool)
assert not adj[0, self_rel][off_diag_mask].any(), "SELF relation has off-diagonal entries"

# for batch 0 (A B A B): turns 0 and 2 are same speaker (A)
# edge 0->2 (past->future, same speaker) -> INTRA_P2F, so adj[b, INTRA_P2F, v=2, u=0] == 1
intra_p2f = _DialogueGCNLayer._INTRA_P2F
assert adj[0, intra_p2f, 2, 0] == 1.0, "intra past->future edge (0->2) missing"

# edge 2->0 (future->past, same speaker) -> INTRA_F2P, adj[b, INTRA_F2P, v=0, u=2] == 1
intra_f2p = _DialogueGCNLayer._INTRA_F2P
assert adj[0, intra_f2p, 0, 2] == 1.0, "intra future->past edge (2->0) missing"

# turns 0 and 1 are different speakers -> inter edges
inter_p2f = _DialogueGCNLayer._INTER_P2F
assert adj[0, inter_p2f, 1, 0] == 1.0, "inter past->future edge (0->1) missing"

# every cell must be binary
assert set(adj.unique().tolist()).issubset({0.0, 1.0}), "adjacency should be binary"

print("PASS _DialogueGCNLayer.build_adjacency")


In [ ]:
D_GCN = 2 * D_FEAT
layer = _DialogueGCNLayer(d_model=D_GCN)

h   = torch.randn(B, N_CTX, D_GCN)
spk = [["A", "B", "A"], ["A", "B", "B"]]   # N_CTX = 3
adj = _DialogueGCNLayer.build_adjacency(spk, N_CTX, device=torch.device("cpu"))
out = layer(h, adj)

assert out.shape == (B, N_CTX, D_GCN), f"expected ({B}, {N_CTX}, {D_GCN}), got {out.shape}"
assert not torch.isnan(out).any(), "NaN in GCN layer output"

print("PASS _DialogueGCNLayer.forward")


In [ ]:
gcn = _DialogueGCN(d_model=D_GCN, num_layers=1)

h   = torch.randn(B, N_CTX, D_GCN)
spk = [["A", "B", "A"], ["A", "B", "B"]]
out = gcn(h, spk)

assert out.shape == (B, N_CTX, D_GCN), f"expected ({B}, {N_CTX}, {D_GCN}), got {out.shape}"
assert not torch.isnan(out).any()

# stacking 2 layers should not change output shape
gcn2 = _DialogueGCN(d_model=D_GCN, num_layers=2)
out2 = gcn2(h, spk)
assert out2.shape == (B, N_CTX, D_GCN), "2-layer GCN shape mismatch"

print("PASS _DialogueGCN")


In [ ]:
import sys
sys.path.insert(0, ".")
sys.path.insert(0, "./capstone_src/style-prompt-generator")

from torch.utils.data import DataLoader
from dataset.ConvoStyleDataset import (
    ConvoStyleDataset, collate_pad
)

H5_PATH   = "capstone_src/data_TEMP/merged_audio_500.h5"
META_PATH = "capstone_src/data_TEMP/merged_metadata_500.parquet"

ds = ConvoStyleDataset(
    h5_path=H5_PATH,
    meta_path=META_PATH,
    meta_columns=["transcription"],
    sample_rate=SAMPLE_RATE,
    num_turns=T,
)
loader = DataLoader(ds, batch_size=B, shuffle=False, collate_fn=collate_pad, num_workers=0)

batch = next(iter(loader))
audio    = batch["audio"]     # (B, T, max_wav_len)
lengths  = batch["lengths"]   # (B, T)
texts    = batch["transcription"]  # list[B][T]

extractor = _UtteranceFeatureExtractor(
    bert_model=real_bert,
    wavlm_model=real_wavlm,
    tokenizer=real_tokenizer,
    processor=real_processor,
    sample_rate=SAMPLE_RATE,
    d_feat=D_FEAT,
)
extractor.eval()

with torch.no_grad():
    feats = extractor(audio, lengths, texts)

assert feats.shape == (B, T, 2 * D_FEAT), f"expected ({B}, {T}, {2*D_FEAT}), got {feats.shape}"

print(f"audio batch shape : {tuple(audio.shape)}")
print(f"feature output    : {tuple(feats.shape)}  (B, T, 2*d_feat)")
for b in range(B):
    for t in range(T):
        L = lengths[b, t].item()
        print(f"  chain {b} turn {t} | {L} samples ({L/SAMPLE_RATE:.2f}s) | text: {texts[b][t][:60]!r}")

print("PASS _UtteranceFeatureExtractor on real dataset batch")


In [ ]:
# reuse: batch, audio, lengths, texts from the dataset cell above
speakers  = batch["speaker_id"]   # list[B][T] str
text_only = batch["text_only"]    # (B, T) bool

D_MODEL  = 128
ATTN_DIM = 64

model = DialogueGraph(
    bert_model=real_bert,
    wavlm_model=real_wavlm,
    tokenizer=real_tokenizer,
    processor=real_processor,
    sample_rate=SAMPLE_RATE,
    d_feat=D_FEAT,
    d_model=D_MODEL,
    d_out=D_OUT,
    attn_dim=ATTN_DIM,
    num_gcn_layers=1,
)
model.eval()

# context_mask marks turns that carry real audio (non-text-only context turns)
context_mask = ~text_only[:, :-1]  # (B, N) where N = T - 1

with torch.no_grad():
    style_vec = model(audio, lengths, texts, speakers, text_only=text_only, context_mask=context_mask)

assert style_vec.shape == (B, D_OUT), f"expected ({B}, {D_OUT}), got {style_vec.shape}"
assert not torch.isnan(style_vec).any(), "NaN in style vector"
assert not torch.isinf(style_vec).any(), "Inf in style vector"

print(f"style_vec shape : {tuple(style_vec.shape)}  (B, d_out)")
print(f"style_vec norm  : {style_vec.norm(dim=-1).tolist()}")
print("PASS DialogueGraph end-to-end on real dataset batch")
